# 01 - Data quality

Checks on the raw GTFS-RT collector output: poll success rate, feed lag
distribution, active trips per hour, share of trips with a usable
`vehicle_id`, and share of stop_time_updates carrying `stop_sequence`.

This notebook reads directly from the Parquet lake at `data/rt/` with
DuckDB and from `logs/collector.log` for poll-level lag and success rate.
It is meant to be re-run as more data accrues; do not hardcode a date range.

In [1]:
import duckdb
import polars as pl
import re
from pathlib import Path

REPO_ROOT = Path.cwd().parent
RT_LAKE = REPO_ROOT / "data" / "rt"
LOG_FILE = REPO_ROOT / "logs" / "collector.log"

con = duckdb.connect()

## Poll success rate and feed lag, parsed from the collector log

In [2]:
poll_re = re.compile(
    r"poll=(\d+) seen=(\d+) written=(\d+) missing_stop_sequence=(\d+) lag_s=(\S+) total_written=(\d+)"
)
fail_re = re.compile(r"poll failed \(fail_count=(\d+)\)")

rows = []
fail_count = 0
with open(LOG_FILE) as f:
    for line in f:
        m = poll_re.search(line)
        if m:
            seen, written, missing, lag, total = m.group(2, 3, 4, 5, 6)
            rows.append(
                {
                    "seen": int(seen),
                    "written": int(written),
                    "missing_stop_sequence": int(missing),
                    "lag_s": None if lag == "None" else int(lag),
                    "total_written": int(total),
                }
            )
        elif fail_re.search(line):
            fail_count += 1

poll_log = pl.DataFrame(rows)
success_rate = poll_log.height / (poll_log.height + fail_count) if (poll_log.height + fail_count) else float("nan")
print(f"successful polls: {poll_log.height}, failed polls: {fail_count}, success rate: {success_rate:.4f}")
poll_log.select("lag_s").describe()

successful polls: 255, failed polls: 3, success rate: 0.9884


statistic,lag_s
str,f64
"""count""",255.0
"""null_count""",0.0
"""mean""",20.305882
"""std""",7.996589
"""min""",2.0
"""25%""",16.0
"""50%""",23.0
"""75%""",24.0
"""max""",38.0


## Active trips per hour and stop_sequence coverage, from the Parquet lake

In [3]:
glob_pattern = str(RT_LAKE / "date=*" / "hour=*" / "*.parquet")

active_trips_per_hour = con.execute(
    f"""
    select
        date_trunc('hour', poll_ts) as hour_bucket,
        count(distinct trip_id) as active_trips,
        count(distinct vehicle_id) as active_vehicles,
        count(*) as row_count
    from read_parquet('{glob_pattern}')
    group by 1
    order by 1
    """
).pl()
active_trips_per_hour

hour_bucket,active_trips,active_vehicles,row_count
"datetime[μs, Europe/Berlin]",i64,i64,i64
2026-07-23 15:00:00 CEST,106564,0,6355672
2026-07-23 16:00:00 CEST,94792,0,5112774
2026-07-23 17:00:00 CEST,91708,0,4773418
2026-07-23 18:00:00 CEST,58649,0,3025002
2026-07-23 19:00:00 CEST,47408,0,2460336
…,…,…,…
2026-07-24 11:00:00 CEST,66387,0,2901171
2026-07-24 12:00:00 CEST,71146,0,3331739
2026-07-24 13:00:00 CEST,73356,0,3248706


In [4]:
vehicle_id_share = con.execute(
    f"""
    select
        count(distinct trip_id) filter (where vehicle_id is not null and vehicle_id != '') as trips_with_vehicle_id,
        count(distinct trip_id) as trips_total
    from read_parquet('{glob_pattern}')
    """
).pl()
vehicle_id_share

trips_with_vehicle_id,trips_total
i64,i64
0,511767


Note: stop_sequence is required to key a row in this collector, so rows
missing it are dropped before they ever reach the lake. The share of dropped
rows is tracked in the collector log via `missing_stop_sequence` above, not
computable from the lake itself.